# Metoda A — Blind spoty: opis (Qwen)

Faza 1 — Qwen opisuje każdy obrazek osobno → CSV z opisami (`descriptions_*.csv`)
Faza 1b — Qwen tworzy opis głównej różnicy na podstawie dwóch opisów → CSV (`diffs_*.csv`)

Używa **wszystkich** rekordów z CSV (bez progu podobieństwa).

## 1. Importy

In [ ]:
import gc
import textwrap
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from IPython.display import display as ipy_display
from PIL import Image
from tqdm.auto import tqdm
from transformers import AutoProcessor, Qwen2VLForConditionalGeneration

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Urządzenie: {DEVICE}")

## 2. Konfiguracja

In [ ]:
SELECTION      = ["clip_caltech_blind_spots.csv", "clip_imagenet_test_blind_spots.csv",
                  "dino_caltech_blind_spots.csv", "dino_imagenet_test_blind_spots.csv",
                  "siglip_caltech_blind_spots.csv", "siglip_imagenet_test_blind_spots.csv"]
# "caltech" | "imagenet" | "all" | lista plików

QWEN_ID        = "Qwen/Qwen2-VL-2B-Instruct"
SIMILARITY_DIR = Path("similarity_results")
CACHE_DIR      = Path("img_cache")
OUTPUT_DIR     = Path("results_A_all_blindspots")
OUTPUT_DIR.mkdir(exist_ok=True)

MAX_NEW_TOKENS_DESC = 500
MAX_NEW_TOKENS_DIFF = 80

## 3. Wczytanie CSV (wszystkie rekordy)

In [ ]:
all_csvs = sorted(SIMILARITY_DIR.glob("*.csv"))

if SELECTION == "caltech":
    selected = [f for f in all_csvs if "caltech" in f.name]
    run_label = "caltech"
elif SELECTION == "imagenet":
    selected = [f for f in all_csvs if "imagenet" in f.name]
    run_label = "imagenet"
elif SELECTION == "all":
    selected = all_csvs
    run_label = "all"
else:
    selected = [SIMILARITY_DIR / f for f in SELECTION]
    stems = [Path(f).stem.replace("_blind_spots", "") for f in SELECTION]
    run_label = "_".join(stems)

desc_path = OUTPUT_DIR / f"descriptions_{run_label}.csv"
diff_path = OUTPUT_DIR / f"diffs_{run_label}.csv"


def dataset_key(csv_path) -> str:
    return "caltech" if "caltech" in str(csv_path) else "imagenet"


print("Wczytuję wszystkie rekordy (bez progu podobieństwa)\n")

# filtered_dfs — nazwa zachowana dla kompatybilności z kolejnymi komórkami
filtered_dfs = {}
total = 0
for csv_path in selected:
    df_all = pd.read_csv(csv_path, sep=";", index_col=0).reset_index(drop=True)
    filtered_dfs[csv_path] = df_all
    total += len(df_all)
    print(f"  {csv_path.name}: {len(df_all)} par")

print(f"\nŁącznie: {total} par")
print(f"Opisy  -> {desc_path}")
print(f"Różnice -> {diff_path}")

## 4. Logowanie HuggingFace

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

## 5. Wczytanie datasetów

In [ ]:
def add_index(sample, idx):
    sample["id"] = idx
    return sample

DATASETS = {}

if any("caltech" in f.name for f in selected):
    print("Wczytywanie Caltech101...")
    DATASETS["caltech"] = load_dataset("flwrlabs/caltech101", split="train")
    print("  Caltech101 OK")

if any("imagenet" in f.name for f in selected):
    print("Wczytywanie ImageNet-1k (test, streaming)...")
    ds = load_dataset("imagenet-1k", split="test", streaming=True)
    DATASETS["imagenet"] = ds.map(add_index, with_indices=True)
    print("  ImageNet-test (streaming) OK")

## 6. Cache obrazków

In [ ]:
def prefetch_pairs(df: pd.DataFrame, key: str) -> None:
    ds = DATASETS[key]
    ids = set(df["id_1"].astype(int)) | set(df["id_2"].astype(int))
    out_dir = CACHE_DIR / key
    out_dir.mkdir(parents=True, exist_ok=True)
    missing = {i for i in ids if not (out_dir / f"{i}.jpg").exists()}
    if not missing:
        print(f"  [{key}] wszystko w cache")
        return
    print(f"  Pobieram {len(missing)} obrazków [{key}]...")
    if key == "caltech":
        for i in tqdm(sorted(missing)):
            ds[i]["image"].convert("RGB").save(out_dir / f"{i}.jpg", "JPEG")
    else:
        max_id = max(missing)
        for cur, sample in enumerate(tqdm(iter(ds), total=max_id + 1)):
            if cur in missing:
                sample["image"].convert("RGB").save(out_dir / f"{cur}.jpg", "JPEG")
                missing.discard(cur)
            if cur >= max_id or not missing:
                break


for csv_path, df_f in filtered_dfs.items():
    if df_f.empty:
        continue
    prefetch_pairs(df_f, dataset_key(csv_path))
print("Cache gotowy.")

## 7. Ładowanie Qwena

In [ ]:
print(f"Ladowanie Qwen ({QWEN_ID})...")
qwen_processor = AutoProcessor.from_pretrained(
    QWEN_ID,
    min_pixels=256 * 28 * 28,
    max_pixels=512 * 28 * 28,
)
qwen_model = Qwen2VLForConditionalGeneration.from_pretrained(
    QWEN_ID, torch_dtype=torch.bfloat16, device_map="auto"
)
qwen_model.eval()
devices = {str(p.device) for p in qwen_model.parameters()}
print(f"  Qwen gotowy | urządzenia: {', '.join(sorted(devices))}")

## 8. Generowanie opisów i zapis do CSV

In [ ]:
DESCRIBE_PROMPT = (
    "Describe this image precisely. Focus on:\n"
    "1. Main subject: exact species, breed, or type; color, size, distinctive markings\n"
    "2. Subject's pose, orientation, and position in the frame\n"
    "3. Background: specific objects present, scene type, environment details\n"
    "4. Overall color palette and lighting conditions\n"
    "Write 3-4 sentences in plain text."
)


def qwen_describe_single(img: Image.Image) -> str:
    content = [{"type": "image", "image": img}, {"type": "text", "text": DESCRIBE_PROMPT}]
    messages = [{"role": "user", "content": content}]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_processor(text=[text], images=[img], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        gen = qwen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS_DESC)
    trimmed = gen[0][inputs.input_ids.shape[1]:]
    del inputs, gen
    torch.cuda.empty_cache()
    return qwen_processor.decode(trimmed, skip_special_tokens=True).strip()


all_results = []
for csv_path, df in filtered_dfs.items():
    if df.empty:
        print(f"\n{csv_path.name} — brak par, pomijam")
        continue
    key = dataset_key(csv_path)
    print(f"\n{csv_path.name}  ({len(df)} par)")
    rows = []
    for _, row in tqdm(df.iterrows(), total=len(df)):
        try:
            img1 = Image.open(CACHE_DIR / key / f"{int(row['id_1'])}.jpg").convert("RGB")
            img2 = Image.open(CACHE_DIR / key / f"{int(row['id_2'])}.jpg").convert("RGB")
            desc1 = qwen_describe_single(img1)
            desc2 = qwen_describe_single(img2)
        except Exception as e:
            print(f"  BLAD {int(row['id_1'])}-{int(row['id_2'])}: {e}")
            desc1, desc2 = "", ""
        result = row.to_dict()
        result.update({"desc_1": desc1, "desc_2": desc2, "source_file": str(csv_path)})
        rows.append(result)
        gc.collect()
    all_results.extend(rows)
    print(f"  {len(rows)} par OK")

desc_df = pd.DataFrame(all_results)
desc_df.to_csv(desc_path, sep=";", index=False)
print(f"\nZapisano: {desc_path}  ({len(desc_df)} wierszy)")
ipy_display(desc_df[["id_1", "id_2", "base_name", "score_base", "desc_1", "desc_2"]].head(3))

## 8b. Podgląd wygenerowanych opisów

In [ ]:
DISPLAY_N_DESC = 20

sample_df = desc_df.sample(n=min(DISPLAY_N_DESC, len(desc_df)), random_state=0).reset_index(drop=True)
print(f"Wyświetlam {len(sample_df)} z {len(desc_df)} par")

for _, row in sample_df.iterrows():
    key = dataset_key(Path(row["source_file"]))
    try:
        img1 = Image.open(CACHE_DIR / key / f"{int(row['id_1'])}.jpg")
        img2 = Image.open(CACHE_DIR / key / f"{int(row['id_2'])}.jpg")
    except FileNotFoundError:
        continue

    fig = plt.figure(figsize=(12, 4.5))
    gs  = gridspec.GridSpec(2, 2, height_ratios=[3, 3], hspace=0.08, wspace=0.05)

    for col, (img, lbl) in enumerate([
        (img1, f"ID {int(row['id_1'])}"),
        (img2, f"ID {int(row['id_2'])}"),
    ]):
        ax = fig.add_subplot(gs[0, col])
        ax.imshow(img)
        ax.set_title(lbl, fontsize=8, pad=2)
        ax.axis("off")

    for col, desc_key in enumerate(["desc_1", "desc_2"]):
        ax_txt = fig.add_subplot(gs[1, col])
        ax_txt.axis("off")
        desc = textwrap.fill(str(row.get(desc_key, ""))[:600], width=55)
        ax_txt.text(
            0.5, 0.97, desc,
            ha="center", va="top", fontsize=6.5,
            bbox=dict(facecolor="#f5f5f5", edgecolor="#ccc", boxstyle="round,pad=0.4"),
            transform=ax_txt.transAxes,
        )

    src = Path(row["source_file"]).name
    plt.suptitle(
        f"{src}  |  {row['base_name']}  |  score: {row['score_base']:.3f}",
        fontsize=8, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    plt.show()
    print()

## 8c. Faza 1b — opis głównej różnicy (tekst)

In [ ]:
DIFF_PROMPT = (
    "[Image 1] {desc_1}\n\n"
    "[Image 2] {desc_2}\n\n"
    "Based on these two descriptions, identify the SINGLE most prominent difference "
    "between the two images. Focus on what stands out most — for example: the main "
    "subjects are different objects or species, the backgrounds show different "
    "environments, or the subject has a different appearance or pose.\n"
    "Describe this main difference in one or two sentences. Be specific and factual."
)


def qwen_diff_text(desc_1: str, desc_2: str) -> str:
    prompt = DIFF_PROMPT.format(desc_1=desc_1, desc_2=desc_2)
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_processor(text=[text], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        gen = qwen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS_DIFF)
    trimmed = gen[0][inputs.input_ids.shape[1]:]
    del inputs, gen
    torch.cuda.empty_cache()
    return qwen_processor.decode(trimmed, skip_special_tokens=True).strip()


diff_results = []
for _, row in tqdm(desc_df.iterrows(), total=len(desc_df)):
    d1 = str(row.get("desc_1", "")).strip()
    d2 = str(row.get("desc_2", "")).strip()
    if not d1 or not d2:
        diff_text = ""
    else:
        try:
            diff_text = qwen_diff_text(d1, d2)
        except Exception as e:
            print(f"  BLAD {int(row['id_1'])}-{int(row['id_2'])}: {e}")
            diff_text = ""
    result = row.to_dict()
    result["diff_text"] = diff_text
    diff_results.append(result)
    gc.collect()

diff_df = pd.DataFrame(diff_results)
diff_df.to_csv(diff_path, sep=";", index=False)
print(f"\nZapisano: {diff_path}  ({len(diff_df)} wierszy)")
ipy_display(diff_df[["id_1", "id_2", "base_name", "diff_text"]].head(3))

## 8d. Podgląd opisów różnic

In [ ]:
DISPLAY_N_DIFF = 10

sample_diff = diff_df.sample(n=min(DISPLAY_N_DIFF, len(diff_df)), random_state=0).reset_index(drop=True)
print(f"Wyświetlam {len(sample_diff)} z {len(diff_df)} par")

for _, row in sample_diff.iterrows():
    key = dataset_key(Path(row["source_file"]))
    try:
        img1 = Image.open(CACHE_DIR / key / f"{int(row['id_1'])}.jpg")
        img2 = Image.open(CACHE_DIR / key / f"{int(row['id_2'])}.jpg")
    except FileNotFoundError:
        continue

    fig = plt.figure(figsize=(12, 5))
    gs  = gridspec.GridSpec(2, 2, height_ratios=[3, 2], hspace=0.08, wspace=0.05)

    for col, (img, lbl) in enumerate([
        (img1, f"ID {int(row['id_1'])}"),
        (img2, f"ID {int(row['id_2'])}"),
    ]):
        ax = fig.add_subplot(gs[0, col])
        ax.imshow(img)
        ax.set_title(lbl, fontsize=8, pad=2)
        ax.axis("off")

    ax_txt = fig.add_subplot(gs[1, :])
    ax_txt.axis("off")
    diff = textwrap.fill(str(row.get("diff_text", ""))[:600], width=110)
    ax_txt.text(
        0.5, 0.97, f"Różnice:\n{diff}",
        ha="center", va="top", fontsize=6.5,
        bbox=dict(facecolor="#f0fff0", edgecolor="#99cc99", boxstyle="round,pad=0.4"),
        transform=ax_txt.transAxes,
    )

    src = Path(row["source_file"]).name
    plt.suptitle(
        f"{src}  |  {row['base_name']}  |  score: {row['score_base']:.3f}",
        fontsize=8, fontweight="bold", y=1.01,
    )
    plt.tight_layout()
    plt.show()
    print()

## 9. Kategoryzacja różnic na podstawie opisu (diff_text)

In [ ]:
CATEGORIES = [
    "SUBJECT_TYPE", "SUBJECT_APPEARANCE", "SUBJECT_POSE",
    "BACKGROUND_OBJECTS", "BACKGROUND_SETTING", "COLORS_LIGHTING", "NO_DIFFERENCE",
]

DIFF_CAT_PROMPT = (
    "The following text describes the main difference between two images:\n\n"
    "\"{diff_text}\"\n\n"
    "Assign exactly ONE category that best matches this difference:\n\n"
    "  - Main subjects are different species, breed, or type? → SUBJECT_TYPE\n"
    "  - Same subject type but different color, markings, or size? → SUBJECT_APPEARANCE\n"
    "  - Same subject, different pose, angle, or orientation? → SUBJECT_POSE\n"
    "  - Same subject, background has different objects present? → BACKGROUND_OBJECTS\n"
    "  - Same subject, background is a different place or environment? → BACKGROUND_SETTING\n"
    "  - Same subject and setting, but different brightness or color tone? → COLORS_LIGHTING\n"
    "  - No meaningful difference? → NO_DIFFERENCE\n\n"
    "Reply in this format:\n"
    "Label: <ONE label>\n"
    "Reason: <one sentence>"
)
MAX_NEW_TOKENS_DIFFCAT = 50


def qwen_categorize_diff(diff_text: str) -> str:
    prompt = DIFF_CAT_PROMPT.format(diff_text=diff_text)
    messages = [{"role": "user", "content": [{"type": "text", "text": prompt}]}]
    text = qwen_processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = qwen_processor(text=[text], return_tensors="pt").to(DEVICE)
    with torch.no_grad():
        gen = qwen_model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS_DIFFCAT)
    trimmed = gen[0][inputs.input_ids.shape[1]:]
    del inputs, gen
    torch.cuda.empty_cache()
    return qwen_processor.decode(trimmed, skip_special_tokens=True).strip()


def parse_category(raw: str) -> str:
    raw_upper = raw.upper()
    for cat in CATEGORIES:
        if cat in raw_upper:
            return cat
    return "UNKNOWN"


diffcat_results = []
for _, row in tqdm(diff_df.iterrows(), total=len(diff_df)):
    dt = str(row.get("diff_text", "")).strip()
    if not dt:
        raw_cat, category = "", "NO_DIFFERENCE"
    else:
        try:
            raw_cat = qwen_categorize_diff(dt)
            category = parse_category(raw_cat)
        except Exception as e:
            print(f"  BLAD {int(row['id_1'])}-{int(row['id_2'])}: {e}")
            raw_cat, category = "", "UNKNOWN"
    result = row.to_dict()
    result["category_raw"] = raw_cat
    result["category"] = category
    diffcat_results.append(result)
    gc.collect()

diffcat_df = pd.DataFrame(diffcat_results)
diffcat_path = diff_path.parent / diff_path.name.replace("diffs_", "categorized_diff_")
diffcat_df.to_csv(diffcat_path, sep=";", index=False)
print(f"\nZapisano: {diffcat_path}  ({len(diffcat_df)} wierszy)")
ipy_display(diffcat_df[["id_1", "id_2", "base_name", "diff_text", "category_raw", "category"]].head(5))

## 10. Rozkład kategorii per model bazowy

In [ ]:
counts = (
    diffcat_df[diffcat_df["category"] != "UNKNOWN"]
    .groupby(["base_name", "category"])
    .size()
    .unstack(fill_value=0)
)

for cat in CATEGORIES:
    if cat not in counts.columns:
        counts[cat] = 0
counts = counts[CATEGORIES]

models = counts.index.tolist()
x = np.arange(len(CATEGORIES))
bar_w = 0.8 / max(len(models), 1)
labels = [c.replace("_", "\n") for c in CATEGORIES]

fig, ax = plt.subplots(figsize=(14, 6))
for i, mdl in enumerate(models):
    offset = (i - len(models) / 2 + 0.5) * bar_w
    ax.bar(x + offset, counts.loc[mdl], bar_w, label=mdl, alpha=0.85)

ax.set_xticks(x)
ax.set_xticklabels(labels, fontsize=9)
ax.set_ylabel("Liczba par", fontsize=11)
ax.legend(title="Model", fontsize=9)
ax.set_title("Rozkład kategorii różnic per model bazowy", fontsize=12, fontweight="bold")
plt.tight_layout()
plt.show()

ipy_display(counts)